**Structured Streaming Basics**

**Topic**: Streaming | Exercises: 8 | Total Time: ~90 min

Practice the Structured Streaming foundation on Databricks: readStream and writeStream against Delta tables, choosing triggers (processingTime vs availableNow), picking the right output mode (append / complete / update), naming queries and reading lastProgress, and deduplicating streams. Every query uses trigger(availableNow=True) and awaitTermination() so the notebook terminates cleanly on Free Edition.

**Solutions**: If stuck, see solutions/structured-streaming-basics-solutions.py for hints and answers.

**Key concepts**:

spark.readStream.table(...) reads a Delta table as a stream of newly committed rows
trigger(availableNow=True) processes everything available and stops (use this on Free Edition)
Output modes: append (only new rows), complete (full result table), update (changed rows)
Streaming queries always need a checkpointLocation for state and offset tracking
.queryName(...) makes a stream identifiable; query.lastProgress exposes runtime metrics
dropDuplicates(["key"]) deduplicates a stream using watermarked state

In [0]:
CATALOG = "db_code"
SCHEMA = "structured_streaming_basics"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

**Exercise 1: Read a Delta Table as a Stream, Write to Another Delta Table**
**Difficulty**: Easy | **Time**: ~10 min

The foundational Structured Streaming pattern: treat an existing Delta table as a stream source and write the streamed rows to a target Delta table in append mode.

**Source **(db_code.structured_streaming_basics.ex1_source): 35 events with columns event_id STRING, event_type STRING, user_id STRING, amount DOUBLE, event_ts TIMESTAMP.

**Target**: Write to db_code.structured_streaming_basics.ex1_target. Expected 35 rows.

**Requirements**:

- Read ex1_source with spark.readStream (Delta source)
- Write to ex1_target as Delta with output mode append
- Set a checkpointLocation under CHECKPOINT_BASE (e.g., f"{CHECKPOINT_BASE}/ex1")
- Use .trigger(availableNow=True) so the stream terminates after one batch
- Call .awaitTermination() so the next cell runs only after the stream finishes

**Constraints**:

Use trigger(availableNow=True) (NOT trigger(once=True), NOT trigger(processingTime=...))
Checkpoint path must be in a Volume

In [0]:
df=spark.readStream.format("delta").table(f"{CATALOG}.{SCHEMA}.ex1_source")

df.writeStream.format("delta")\
    .outputMode("append")\
    .option('checkpointLocation',f"{CHECKPOINT_BASE}/ex_1")\
    .trigger(availableNow=True)\
    .toTable(f"{CATALOG}.{SCHEMA}.ex1_target")\
    .awaitTermination()    



In [0]:
# Validate Exercise 1
result = spark.table(f"{CATALOG}.{SCHEMA}.ex1_target")

assert result.count() == 35, f"Expected 35 rows, got {result.count()}"
assert "event_id" in result.columns, "Missing event_id column"
assert "amount" in result.columns, "Missing amount column"
assert result.filter("event_id = 'EV-001'").count() == 1, "EV-001 should exist"
assert result.filter("event_id = 'EV-035'").count() == 1, "EV-035 should exist"
ev3_amount = result.filter("event_id = 'EV-003'").select("amount").collect()[0][0]
assert ev3_amount == 120.50, f"EV-003 amount should be 120.50, got {ev3_amount}"

print("Exercise 1 passed!")

**Exercise 2: Filter a Streaming DataFrame Before Writing**

**Difficulty**: Easy | **Time**: ~10 min

Transformations on streaming DataFrames work exactly like batch transformations - filters, projections, and column derivations all compose into the streaming plan. Filter the events stream to keep only purchase events before writing.

**Source **(db_code.structured_streaming_basics.ex2_source): 35 events. 18 rows have event_type = 'purchase', the rest are view or click.

**Target**: Write to db_code.structured_streaming_basics.ex2_target. Expected 18 rows, all with event_type = 'purchase'.

**Requirements**:

- Read ex2_source with spark.readStream
- Keep only rows where event_type = 'purchase'
- Write to ex2_target as Delta in append mode
- Use .trigger(availableNow=True) and .awaitTermination()
- Checkpoint at f"{CHECKPOINT_BASE}/ex2"

**Constraints**:

- Apply the filter on the streaming DataFrame, not as a post-write SQL query
- Output must contain ONLY purchase events

In [0]:
from pyspark.sql import functions as F

df_2=spark.readStream.format("delta").table(f"{CATALOG}.{SCHEMA}.ex2_source").filter(F.col("event_type")=='purchase')


df_2.writeStream.format('delta')\
    .outputMode('append')\
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex_2")\
    .trigger(availableNow=True)\
    .toTable(f'{CATALOG}.{SCHEMA}.ex2_target')\
    .awaitTermination()            


In [0]:
# Validate Exercise 2
result = spark.table(f"{CATALOG}.{SCHEMA}.ex2_target")

assert result.count() == 18, f"Expected 18 rows, got {result.count()}"
non_purchase = result.filter("event_type != 'purchase'").count()
assert non_purchase == 0, f"Expected 0 non-purchase rows, got {non_purchase}"
assert result.filter("event_id = 'EV-001'").count() == 1, "EV-001 (purchase) should be present"
assert result.filter("event_id = 'EV-002'").count() == 0, "EV-002 (view) should be filtered out"

print("Exercise 2 passed!")

**Exercise 3: Add a Derived Column to a Streaming DataFrame**

**Difficulty**: Easy | **Time**: ~10 min

Streaming DataFrames support withColumn just like batch DataFrames. Add a derived column amount_with_tax equal to amount * 1.10 (10% tax) before writing.

Source (db_code.structured_streaming_basics.ex3_source): 35 events with the standard schema. amount is DOUBLE.

**Target**: Write to db_code.structured_streaming_basics.ex3_target. Expected 35 rows. The target must include all original columns PLUS an amount_with_tax DOUBLE column.

**Requirements:**

- Read ex3_source with spark.readStream
- Add a new column amount_with_tax = amount * 1.10
- Write to ex3_target as Delta in append mode
- Use .trigger(availableNow=True) and .awaitTermination()
- Checkpoint at f"{CHECKPOINT_BASE}/ex3"

**Constraints:**

- The new column must be named exactly amount_with_tax
- The original amount column must still be present


In [0]:
from pyspark.sql import functions as F

df=spark.readStream.format("delta").table(f"{CATALOG}.{SCHEMA}.ex3_source").withColumn('amount_with_tax',F.col("amount")*1.10)

df.writeStream.format('delta')\
    .outputMode("append")\
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex_3")\
    .trigger(availableNow=True)\
    .toTable(f"{CATALOG}.{SCHEMA}.ex3_target")\
    .awaitTermination()   

In [0]:
# Validate Exercise 3
result = spark.table(f"{CATALOG}.{SCHEMA}.ex3_target")

assert result.count() == 35, f"Expected 35 rows, got {result.count()}"
assert "amount_with_tax" in result.columns, "Missing amount_with_tax column"
assert "amount" in result.columns, "Original amount column must still be present"

# EV-001 has amount=50.00, so amount_with_tax should be 55.00
ev1_tax = result.filter("event_id = 'EV-001'").select("amount_with_tax").collect()[0][0]
assert abs(ev1_tax - 55.00) < 1e-6, f"EV-001 amount_with_tax should be 55.00, got {ev1_tax}"
# EV-003 has amount=120.50, so amount_with_tax should be 132.55
ev3_tax = result.filter("event_id = 'EV-003'").select("amount_with_tax").collect()[0][0]
assert abs(ev3_tax - 132.55) < 1e-6, f"EV-003 amount_with_tax should be 132.55, got {ev3_tax}"

print("Exercise 3 passed!")

**Exercise 4**: Append-Only Ingestion - Pick the Right Output Mode
**Difficulty**: Medium | **Time**: ~10 min

Output mode is one of append, complete, or update. Each is only legal for certain query shapes. For an ingestion sink that simply copies streamed events into a target, only ONE mode is correct - the other two fail. This exercise has you write the pipeline AND record which mode is correct (and which two would fail and why).

Source (db_code.structured_streaming_basics.ex4_source): 35 events.

**Target**: Write to db_code.structured_streaming_basics.ex4_target. Expected 35 rows.

**Requirements:**

- Read ex4_source and write it unchanged to ex4_target
- Pass the correct output mode to outputMode(...) on the writeStream
- Use .trigger(availableNow=True) and .awaitTermination()
- Checkpoint at f"{CHECKPOINT_BASE}/ex4"
- Set a Python variable correct_mode to the string of the mode you used (one of "append", "complete", "update")

Constraints:

- The pipeline has no aggregation, so complete and update are illegal here
- You must call outputMode(...) explicitly even if the default would work

In [0]:
df=spark.readStream.format("delta").table(f"{CATALOG}.{SCHEMA}.ex4_source")

correct_mode="append"

df.writeStream.format('delta')\
    .outputMode(correct_mode)\
    .option("checkpointLocation",f"{CHECKPOINT_BASE}/ex_4")\
    .trigger(availableNow=True)\
    .toTable(f"{CATALOG}.{SCHEMA}.ex4_target")\
    .awaitTermination()

In [0]:
# Validate Exercise 4
result = spark.table(f"{CATALOG}.{SCHEMA}.ex4_target")

assert result.count() == 35, f"Expected 35 rows, got {result.count()}"
assert "correct_mode" in dir(), "Set a variable `correct_mode` to your chosen output mode"
assert correct_mode == "append", \
    f"For an append-only ingestion sink (no aggregation), the correct mode is 'append'. Got '{correct_mode}'"
# Verify a specific value to catch wrong data
ev5_amount = result.filter("event_id = 'EV-005'").select("amount").collect()[0][0]
assert ev5_amount == 75.25, f"EV-005 amount should be 75.25, got {ev5_amount}"

print("Exercise 4 passed!")

**Exercise 5: Trigger Modes - Understanding the Cost/Latency Tradeoff**
**Difficulty**: Medium | **Time**: ~10 min

Triggers control when a micro-batch fires. Two modes you must know:

processingTime='30 seconds': long-running query, fires every 30 seconds, keeps compute warm. Lower latency, higher cost.
availableNow=True: processes everything currently available, then stops. Compute spins down between scheduled runs. Higher latency, lower cost.
Free Edition note: Trigger.ProcessingTime is NOT supported on Free Edition (which is serverless-only). Per the Databricks docs: "On serverless compute, only Trigger.AvailableNow() and Trigger.Once() are supported." You would only see processingTime in production on classic compute. In this exercise you write the availableNow query and identify which trigger wins on latency and which wins on cost.

Source (db_code.structured_streaming_basics.ex5_source): 35 events.

**Target** (ex5_target_available_now): 35 rows written by the availableNow query.

**Requirements:**

- Write a streaming query with .trigger(availableNow=True), sink to ex5_target_available_now, checkpoint f"{CHECKPOINT_BASE}/ex5_an", then .awaitTermination().
- Set a Python variable latency_winner to the trigger string that gives LOWER LATENCY (one of "processingTime" or "availableNow").
- Set a Python variable cost_winner to the trigger string that minimizes COST on Free Edition / scheduled jobs (the one where compute spins down between runs).

**Constraints**:

- Do NOT use .trigger(processingTime=...) here - it would fail on Free Edition serverless.
- Use a checkpoint location in a Volume.

In [0]:
(spark.readStream
    .table(f"{CATALOG}.{SCHEMA}.ex5_source")
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex5_an")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.ex5_target_available_now")
    .awaitTermination()
)

latency_winner = "processingTime"  # continuous compute -> lower per-batch latency
cost_winner = "availableNow"        # compute spins down between runs -> lower cost

In [0]:
# Validate Exercise 5
an = spark.table(f"{CATALOG}.{SCHEMA}.ex5_target_available_now")

assert an.count() == 35, f"availableNow target should have 35 rows, got {an.count()}"

assert "latency_winner" in dir(), "Set `latency_winner` to 'processingTime' or 'availableNow'"
assert "cost_winner" in dir(), "Set `cost_winner` to 'processingTime' or 'availableNow'"
assert latency_winner == "processingTime", \
    f"processingTime gives lower latency (continuous compute). Got '{latency_winner}'"
assert cost_winner == "availableNow", \
    f"availableNow minimizes cost (compute spins down between runs). Got '{cost_winner}'"

print("Exercise 5 passed!")

**Exercise 6: Streaming Aggregation in Complete Output Mode**

**Difficulty**: Medium | **Time**: ~10 min

Stateful aggregations (groupBy without windowing) require complete output mode when writing to a Delta sink without a watermark - the engine emits the FULL result table after every micro-batch, overwriting the previous result. This is fine for small grouping dimensions (handful of event types). For large key spaces, you would add a watermark and switch to update.

**Source** (db_code.structured_streaming_basics.ex6_source): 35 events with 3 distinct event_type values: purchase (18), view (9), click (8).

**Target**: Write to db_code.structured_streaming_basics.ex6_target. Expected 3 rows, one per event_type, with a column cnt holding the count.

**Requirements:**

- Read ex6_source with spark.readStream
- Group by event_type and compute count(*) AS cnt
- Write to ex6_target with outputMode("complete")
- Use .trigger(availableNow=True) and .awaitTermination()
- Checkpoint at f"{CHECKPOINT_BASE}/ex6"

**Constraints:**

- The count column MUST be named cnt (the validator checks this name)
- Use complete mode - append would fail because there is no watermark, and the result table is being overwritten on every batch

In [0]:
from pyspark.sql import functions as F

df=spark.readStream.format('delta').table(f"{CATALOG}.{SCHEMA}.ex6_source").groupBy("event_type").agg(F.count(F.col("event_id")).alias("cnt"))

df.writeStream.format("delta")\
  .outputMode("complete")\
  .option("checkpointLocation",f"{CHECKPOINT_BASE}/ex_6")\
  .trigger(availableNow=True).toTable(f"{CATALOG}.{SCHEMA}.ex6_target")\
  .awaitTermination()






In [0]:
# Validate Exercise 6
result = spark.table(f"{CATALOG}.{SCHEMA}.ex6_target")

assert result.count() == 3, f"Expected 3 rows (one per event_type), got {result.count()}"
assert "event_type" in result.columns, "Missing event_type column"
assert "cnt" in result.columns, "Count column must be named 'cnt'"

purchase_cnt = result.filter("event_type = 'purchase'").select("cnt").collect()[0][0]
view_cnt = result.filter("event_type = 'view'").select("cnt").collect()[0][0]
click_cnt = result.filter("event_type = 'click'").select("cnt").collect()[0][0]

assert purchase_cnt == 18, f"purchase count should be 18, got {purchase_cnt}"
assert view_cnt == 9, f"view count should be 9, got {view_cnt}"
assert click_cnt == 8, f"click count should be 8, got {click_cnt}"

print("Exercise 6 passed!")

**Exercise 7: Name a Query and Inspect lastProgress**

**Difficulty**: Medium | **Time**: ~10 min

Production streams need observability. .queryName(...) gives a stream a human-readable identifier; query.lastProgress exposes the most recent batch's metrics (numInputRows, processedRowsPerSecond, inputRowsPerSecond, batch durations, etc.). In this exercise you start a named streaming write, wait for it to finish, then read lastProgress to capture metrics and store them in a Delta metrics table.

Source (db_code.structured_streaming_basics.ex7_source): 35 events.

**Targets:**

**ex7_target: written by the stream (35 rows)
ex7_progress: 1 row with columns query_name STRING, num_input_rows BIGINT**

**Requirements:**

- Start a streaming write from ex7_source to ex7_target with .queryName("ss_basics_ex7")
- Use .trigger(availableNow=True) and .awaitTermination()
- Checkpoint at f"{CHECKPOINT_BASE}/ex7"
- After the stream terminates, read query.lastProgress and extract the query name and numInputRows. Write these as a 1-row Delta table ex7_progress with columns query_name STRING, num_input_rows BIGINT.

**Constraints:**

- The query name string MUST be ss_basics_ex7
- Use lastProgress, not recentProgress (the validator checks the single-batch row)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

query = (spark.readStream
    .table(f"{CATALOG}.{SCHEMA}.ex7_source")
    .writeStream
    .queryName("ss_basics_ex7")
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex7")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.ex7_target")
)
query.awaitTermination()

progress = query.lastProgress
metrics_schema = StructType([
    StructField("query_name", StringType()),
    StructField("num_input_rows", LongType()),
])
metrics_df = spark.createDataFrame(
    [(progress["name"], int(progress["sources"][0]["numInputRows"]))],
    schema=metrics_schema,
)
metrics_df.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.ex7_progress"
)

In [0]:
progress = query.lastProgress
metrics_schema = StructType([
    StructField("query_name", StringType()),
    StructField("num_input_rows", LongType()),
])
metrics_df = spark.createDataFrame(
    [(progress["name"], int(progress["sources"][0]["numInputRows"]))],
    schema=metrics_schema,
)
metrics_df.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.ex7_progress"
)

In [0]:
progress = query.lastProgress
progress

In [0]:
progress["sources"][0]["numInputRows"]

In [0]:
# Validate Exercise 7
target = spark.table(f"{CATALOG}.{SCHEMA}.ex7_target")
progress = spark.table(f"{CATALOG}.{SCHEMA}.ex7_progress")

assert target.count() == 35, f"ex7_target should have 35 rows, got {target.count()}"
assert progress.count() == 1, f"ex7_progress should have exactly 1 row, got {progress.count()}"

row = progress.collect()[0]
assert row["query_name"] == "ss_basics_ex7", \
    f"query_name should be 'ss_basics_ex7', got '{row['query_name']}'"
# Total numInputRows across the single availableNow batch must equal source row count
assert row["num_input_rows"] == 35, \
    f"num_input_rows should be 35 (rows in source), got {row['num_input_rows']}"

print("Exercise 7 passed!")

**Exercise 8: Deduplicate a Stream with dropDuplicates**
**Difficulty**: Hard | **Time:** ~20 min

Exactly-once delivery is a common requirement: even if the source delivers an event multiple times (retries, replay, upstream bug), the target must contain each event_id exactly once. dropDuplicates(["event_id"]) on a streaming DataFrame stores seen keys in state and drops duplicates on subsequent batches.

**Source** (db_code.structured_streaming_basics.ex8_source): 15 rows with deliberate duplicates -

EV-101 appears 3 times
EV-102, EV-104, EV-107 each appear 2 times
The remaining 6 event_ids appear once
Total unique event_ids: 10
Target: Write to db_code.structured_streaming_basics.ex8_target. Expected 10 rows, one per unique event_id. No duplicate event_id may appear.

**Requirements:**

Read ex8_source with spark.readStream
Apply dropDuplicates(["event_id"]) on the streaming DataFrame
Write to ex8_target as Delta in append mode
Use .trigger(availableNow=True) and .awaitTermination()
Checkpoint at f"{CHECKPOINT_BASE}/ex8"

**Constraints:**

Use dropDuplicates on the STREAMING DataFrame (not a post-write SELECT DISTINCT)
Dedup key is event_id only

In [0]:
# EXERCISE_KEY: ss_basics_ex8
(spark.readStream
    .table(f"{CATALOG}.{SCHEMA}.ex8_source")
    .dropDuplicates(["event_id"])
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex8")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.ex8_target")
    .awaitTermination()
)

In [0]:
# Validate Exercise 8
result = spark.table(f"{CATALOG}.{SCHEMA}.ex8_target")

assert result.count() == 10, f"Expected 10 unique rows, got {result.count()}"
# Uniqueness check
distinct_ids = result.select("event_id").distinct().count()
assert distinct_ids == 10, f"Expected 10 distinct event_ids, got {distinct_ids}"
# No duplicate event_id allowed
assert result.count() == distinct_ids, \
    f"Row count ({result.count()}) does not match distinct event_id count ({distinct_ids})"
# Spot checks - duplicated source rows must appear exactly once
assert result.filter("event_id = 'EV-101'").count() == 1, "EV-101 should appear exactly once after dedup"
assert result.filter("event_id = 'EV-107'").count() == 1, "EV-107 should appear exactly once after dedup"
assert result.filter("event_id = 'EV-110'").count() == 1, "EV-110 should appear exactly once"

print("Exercise 8 passed!")

In [0]:
%sql
select * from db_code.structured_streaming_basics.ex6_target;

In [0]:
%sql
select * from db_code.structured_streaming_basics.ex7_progress